# Clase 5 - Integracion: de ocho etapas a un sistema que decide

**Pregunta de la clase:** *¿cómo se arma un sistema de visión completo —de la
cámara a la decisión— encadenando lo de las clases 1 a 4, sin introducir nada
nuevo?*

Las ocho etapas ya se vieron, cada una en su clase. Lo nuevo de hoy es
encadenarlas y hacerlas trabajar juntas: ADQUISICION -> PREPROCESAMIENTO ->
SEGMENTACION/DETECCION -> EXTRACCION -> ML/DL -> ANALISIS -> VISUALIZACION ->
INTERACCION. Este cuaderno corre los cinco sistemas de referencia del
repositorio y mide lo que cada uno decide.

## Objetivos

Al terminar el cuaderno, el estudiante debe ser capaz de:

1. Recorrer la cadena completa con un sistema que ejecuta de principio a fin.
2. Explicar qué recibe y qué devuelve cada etapa: el contrato entre etapas.
3. Medir el sistema como sistema: tiempo por etapa, coste de cada error (FN
   vs. FP) y la cifra que decide.
4. Sostener el dominio elegido con el material de las clases 1 a 4.
5. Insertar una variación en UNA etapa y medir su consecuencia en la cifra.
6. Presentar el sistema en 5 minutos: una figura y una cifra.

## Preparacion

El cuaderno corre dentro del repositorio; si se abre en Colab sin el
repositorio, la primera celda detecta dónde está el curso o degrada con un
mensaje. Los cinco sistemas de referencia se ejecutan con la consola
headless (SDL dummy): nada abre una ventana aquí.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CURSO = Path.cwd()
for candidata in (CURSO, CURSO.parent, CURSO / 'computer-vision-course', CURSO.parent / 'computer-vision-course'):
    if (candidata / 'cvcourse').exists():
        CURSO = candidata
        break
else:
    raise RuntimeError(f'no encuentro el curso desde {CURSO}')
if str(CURSO) not in sys.path:
    sys.path.insert(0, str(CURSO))

from cvcourse import features, synthetic

SEMILLA = 42
HEADLESS = {**os.environ, 'SDL_VIDEODRIVER': 'dummy', 'SDL_AUDIODRIVER': 'dummy',
           'PYGAME_HIDE_SUPPORT_PROMPT': '1', 'MPLBACKEND': 'Agg'}
print('listo. Curso en', CURSO)

## El experimento

### T1 - Correr los cinco sistemas de referencia

La cadena como arquitectura primero, en una tabla que hay que poder
reproducir de memoria: cada etapa recibe una cosa y devuelve otra.

| Etapa | Recibe | Devuelve |
|---|---|---|
| 1. Adquisicion | fuente (camara, fichero, motor, generador) | imagen |
| 2. Preprocesamiento | imagen | imagen limpia |
| 3. Segmentacion/deteccion | imagen limpia | mascara / regiones |
| 4. Extraccion | regiones | tabla de caracteristicas |
| 5. ML/DL | caracteristicas | prediccion + probabilidad |
| 6. Analisis | predicciones | decision + su coste |
| 7. Visualizacion | etapas | figuras |
| 8. Interaccion | decision | accion o reporte |

Ahora se ejecutan los cinco sistemas del curso, uno por dominio. Mientras
corren, anota en cada uno: (a) qué pregunta responde, (b) cuál es la cifra
que decide, (c) qué pieza de las clases 1-4 tomó cada etapa.

In [ ]:
SISTEMAS = {
    'industrial  (estacion de inspeccion)  ': 'examples/class05_integration/industrial/estacion_de_inspeccion.py',
    'videojuego  (analizador de fotogramas) ': 'examples/class05_integration/game/analizador_de_fotogramas.py',
    'mecatronica (percepcion pick-and-place)': 'examples/class05_integration/mechatronics/percepcion_pick_and_place.py',
    'manufactura (linea de conteo)          ': 'examples/class05_integration/manufacturing/linea_de_conteo.py',
    'datos       (panel de resultados)      ': 'examples/class05_integration/data_analysis/panel_de_resultados.py',
}

def correr_sistema(etiqueta, ruta):
    r = subprocess.run([sys.executable, str(CURSO / ruta)],
                       capture_output=True, text=True, encoding='utf-8',
                       errors='replace', env=HEADLESS, cwd=str(CURSO),
                       timeout=600)
    print('=' * 28, etiqueta)
    salida = r.stdout if r.returncode == 0 else r.stderr
    for linea in salida.splitlines()[-6:]:
        print('   ', linea.strip())
    print(f'   [exit {r.returncode}]')
    return r

In [ ]:
resultados = {nombre: correr_sistema(nombre, ruta)
              for nombre, ruta in SISTEMAS.items()}
print('\nLos cinco sistemas ejecutaron. Anota dominio, cifra y piezas 1-4 en T2.')

### T2 - Lo que cada sistema decide y la cifra que lo respalda

Cada sistema existe para responder UNA pregunta con UNA cifra. Completar la
tabla con lo que se acaba de medir; una cifra que no sale de la salida de
T1 no vale.

In [ ]:
CIFRAS_DE_LOS_SISTEMAS = {
    'industrial': 'coste del turno: FN*20 + FP*1 acumulado en el reporte',
    'videojuego': 'alerta por fotograma: player -> IGNORAR, enemies/bosses -> ALERTA',
    'mecatronica': 'orden de agarre en milimetros para cada pieza NO_OK',
    'manufactura': 'conteo de la banda: 6/6 piezas, 1 expulsada (NO_OK)',
    'datos': 'tabla de modelos honesta; el clasificador se elige por sus celdas',
}
for dominio, cifra in CIFRAS_DE_LOS_SISTEMAS.items():
    print(f'{dominio:13s} -> {cifra}')

### T3 - El contrato entre etapas: la leccion de la clase

La extraccion mide la MASCARA de la segmentacion, no la imagen de nuevo:
recibe regiones y devuelve una fila por region. Si la segmentacion empieza a
entregar tres regiones donde habia una, la extraccion no avisa: mide tres
piezas y el modelo las clasifica — mal o bien, pero sin avisar. La prueba
de abajo lo muestra en una sola pieza defectuosa: la grieta la parte en el
umbral de Otsu.

In [ ]:
# Una pieza con grieta, como la de la mesa del pick-and-place.
imagen, verdad = synthetic.pieza_individual(tamano=128, defecto='grieta', semilla=1)
gris = imagen[..., :3].mean(axis=2).astype(np.uint8) if imagen.ndim == 3 else imagen.astype(np.uint8)
_, mascara = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
n_sin, _ = cv2.connectedComponents(mascara.astype(np.uint8))
print('componentes con Otsu solo      :', n_sin - 1, '(la grieta parte la pieza)')

cierre = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
n_con, _ = cv2.connectedComponents(cierre.astype(np.uint8))
print('componentes con cierre 3x3     :', n_con - 1)

rotas = features.caracteristicas_de_mascara(mascara > 0, etiqueta_de_clase=verdad.clase)
enteras = features.caracteristicas_de_mascara(cierre > 0, etiqueta_de_clase=verdad.clase)
print('filas que recibe el modelo     :', len(rotas), 'sin cierre |', len(enteras), 'con cierre')
print('La extraccion no avisa: mide lo que recibe. Por eso el contrato de')
print('cada etapa se declara en la firma y se mide en la salida.')

### T4 - El coste de cada error segun el dominio

La pareja FN/FP no es abstracta: cada dominio la paga con distinta moneda.
En la estacion industrial una pieza mala que sale al cliente cuesta 20 y una
buena rechazada cuesta 1; en la linea de conteo expulsar de mas ni siquiera
se nota en el conteo; en el juego, clasificar mal al boss es perder la
partida. Escribir debajo la pareja de TU dominio futuro con UN numero que la
haga discutible.

## Reto (guia §5.5)

Insertar una variacion en UNA etapa y medir la consecuencia en la cifra
final. Aqui la variacion es la etapa 1: la adquisicion entrega el mismo
lote con mas ruido. El modelo (etapa 5) no se reentrena: se evalua sobre
piezas que no entrenaron, igual que en la Clase 4. Si la cifra se mueve, el
analisis lo dice; si no se mueve, tambien es una medicion.

In [ ]:
def filas_de(imagenes, etiquetas):
    filas = []
    for imagen, etiqueta in zip(imagenes, etiquetas, strict=True):
        gris = imagen.astype(np.uint8)
        _, mascara = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        filas.extend(features.caracteristicas_de_mascara(mascara > 0, etiqueta_de_clase=etiqueta))
    return filas

imgs_bajo, v_bajo = synthetic.lote_de_piezas(n=120, semilla=20260805, ruido=3.0)
imgs_alto, v_alto = synthetic.lote_de_piezas(n=120, semilla=20260805, ruido=80.0)
etiquetas = [v.clase for v in v_bajo]
X_bajo, y_bajo, _ = features.a_matriz(filas_de(imgs_bajo, etiquetas))
X_alto, y_alto, _ = features.a_matriz(filas_de(imgs_alto, etiquetas))

# La misma particion para ambos lotes: son las mismas piezas, otro ruido.
idx_tr, idx_te = train_test_split(range(len(X_bajo)), test_size=0.3, random_state=SEMILLA, stratify=y_bajo)
X_tr, y_tr = X_bajo[idx_tr], y_bajo[idx_tr]

knn = Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())])
knn.fit(X_tr, y_tr)
print('adquisicion con ruido bajo (sigma 3) : acc', round(knn.score(X_bajo[idx_te], y_bajo[idx_te]), 3))
print('adquisicion con ruido alto (sigma 80): acc', round(knn.score(X_alto[idx_te], y_alto[idx_te]), 3))
print('Variacion en UNA etapa, consecuencia medida en la cifra final.')

## Preguntas de análisis

Responder cada una con una cifra o una figura de este cuaderno:

1. En la escena de la pieza con grieta (T3): si la etapa 3 cambiara y la
   extraccion recibiera la pieza rota en dos, ¿que pasa con las filas del
   modelo y con la decision final? ¿Que etapa se reentrena y cual no?
2. En el reto: ¿cuanto cambio la cifra al subir el ruido de 3 a 80? Si esa
   camara nueva costara el doble en cada fotograma, ¿donde se nota primero
   en la cadena: en la precision o en el tiempo por etapa?
3. ¿Que etapa de los cinco sistemas fue la mas cara de integrar y que
   contrato hubo que aclarar? (La linea de conteo reajusto su modelo:
   ¿por que?)
4. ¿Que pasaria si la adquisicion cambiara de verdad (otra camara, otro
   fondo, otro nivel del juego)? ¿Que etapa absorbe el cambio y cual se
   rompe primero?
5. Cita una pieza concreta de las clases 1-4 dentro de cada uno de los cinco
   sistemas (ruta del ejemplo, no 'lo visto en clase').

## Conclusiones

Integrar no es sumar bibliotecas: es encadenar contratos. Un sistema de
vision se valida con su cifra final, y por eso cada etapa se mide: si la
cifra cambia cuando una etapa cambia, el sistema esta vivo; si no, la
etapa no hace nada o la cifra miente. Con una figura y una cifra se
presenta el proyecto del grupo en 5 minutos.

## Bibliografía

- Guia de la clase: `docs/clase05_guia.md` (contratos, rúbrica, dominios).
- Ejemplos de referencia: `examples/class05_integration/{industrial,game,mechatronics,manufacturing,data_analysis}/`.
- Piezas reutilizadas: `examples/class03_segmentation/`, `examples/class04_ml_dl/`.
- Solucion de referencia: `solutions/clase05_solucion.py`.